# STAGE 6: FusionLand AI Training (Google Colab)

This notebook performs the final 50-epoch training for the MultimodalFusionNet on Google Colab.

**Instructions:**
1. Go to **Runtime -> Change runtime type** and select **T4 GPU** (or L4/V100).
2. Run all cells.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Navigate to the project root directory
os.chdir('/content/drive/MyDrive/GeoFusion AI')
print(f"Current working directory: {os.getcwd()}")

In [ ]:
# Install project dependencies
!pip install -r requirements.txt -q

In [ ]:
import sys
import torch

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Device:", torch.device("cuda"))
else:
    print("ERROR: CUDA is not available! Please enable a Colab GPU runtime.")
    sys.exit("Stopping execution because GPU is required for STAGE 6 Training.")

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader
from tqdm import tqdm

# Import project modules
from src.utils.config import BATCH_SIZE, LEARNING_RATE, MODELS_DIR, SPLITS_DIR, RANDOM_SEED
from src.models.config import NUM_CLASSES
from src.training.dataset import MultimodalSatelliteDataset
from src.training.trainer import ModelTrainer

# Training Configuration
EPOCHS = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(RANDOM_SEED)

# File validations
train_csv = SPLITS_DIR / "train.csv"
val_csv = SPLITS_DIR / "val.csv"
test_csv = SPLITS_DIR / "test.csv"

assert train_csv.exists(), "Train CSV not found!"
assert val_csv.exists(), "Val CSV not found!"
assert test_csv.exists(), "Test CSV not found! (Used for post-training eval only)"

print("Loading datasets...")
train_dataset = MultimodalSatelliteDataset(split_csv_path=train_csv)
val_dataset = MultimodalSatelliteDataset(split_csv_path=val_csv)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Loaded {len(train_dataset)} training samples and {len(val_dataset)} validation samples.")

In [ ]:
print("Calculating class weights from training set to handle imbalance...")
class_counts = np.zeros(NUM_CLASSES)
for _, _, labels in tqdm(train_loader, desc="Counting classes"):
    unique, counts = torch.unique(labels, return_counts=True)
    for u, c in zip(unique.tolist(), counts.tolist()):
        class_counts[u] += c

total_pixels = class_counts.sum()
class_weights = total_pixels / (NUM_CLASSES * class_counts)
class_weights = np.nan_to_num(class_weights, posinf=1.0, neginf=1.0)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
print(f"\nComputed Class Weights: {class_weights_tensor.tolist()}")

print("\nRunning one-batch sanity check...")
s1_batch, s2_batch, lbl_batch = next(iter(train_loader))
print(f"S1 batch shape: {s1_batch.shape}")
print(f"S2 batch shape: {s2_batch.shape}")
print(f"Labels batch shape: {lbl_batch.shape}")

In [ ]:
# Initialize Trainer (natively uses AdamW and Early Stopping via ReduceLROnPlateau)
trainer = ModelTrainer(
    model_type="fusion",
    num_classes=NUM_CLASSES,
    lr=LEARNING_RATE,
    save_dir=MODELS_DIR,
    device=device,
    class_weights=class_weights_tensor
)

checkpoint_name = "fusion_model_best.pth"
print("\n" + "=" * 52)
print(f"=== TRAINING FUSION MODEL FOR {EPOCHS} EPOCHS ===")
print("=" * 52)

# Start Training Loop
history = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=EPOCHS,
    checkpoint_name=checkpoint_name,
)

In [ ]:
# Save Artifacts and Generate Report
BASE_DIR = Path(".").resolve()
outputs_training_dir = BASE_DIR / "outputs" / "training"
outputs_training_dir.mkdir(parents=True, exist_ok=True)

history_path = outputs_training_dir / "training_history.json"
with open(history_path, "w", encoding="utf-8") as f:
    json.dump(history, f, indent=4)
    
# Plotting
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
epochs_range = range(1, len(history["train_loss"]) + 1)
plt.plot(epochs_range, history["train_loss"], label="Train Loss", linestyle="--")
plt.plot(epochs_range, history["val_loss"], label="Val Loss", linestyle="-")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, [acc * 100 for acc in history["train_acc"]], label="Train Acc", linestyle="--")
plt.plot(epochs_range, [acc * 100 for acc in history["val_acc"]], label="Val Acc", linestyle="-")
plt.xlabel("Epochs")
plt.ylabel("Pixel Accuracy (%)")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
graph_path = outputs_training_dir / "loss_accuracy_graph.png"
plt.savefig(graph_path, dpi=300)
plt.show()
plt.close()

best_val_loss = min(history["val_loss"])
best_val_acc = max(history["val_acc"]) * 100

print("\n" + "=" * 52)
print("FINAL STAGE 6 TRAINING COMPLETED")
print("=" * 52)
print(f"- GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"- CUDA availability: {torch.cuda.is_available()}")
print(f"- number of epochs completed: {len(history['train_loss'])}")
print(f"- best validation loss: {best_val_loss:.4f}")
print(f"- best validation accuracy: {best_val_acc:.2f}%")
print(f"- checkpoint path: {MODELS_DIR / checkpoint_name}")
print(f"- training history path: {history_path}")
print(f"- graph path: {graph_path}")